In [2]:
# Setup correct venv
#!pip install "plotly<6.0.0" ##---> workaround titleside error

In [3]:
from sklearn.preprocessing import FunctionTransformer
from pathlib import Path
from joblib import load
import utils
import pandas as pd
from umap import UMAP

from gtda.mapper import (
    CubicalCover,
    make_mapper_pipeline,
    plot_static_mapper_graph,
    plot_interactive_mapper_graph
)

from sklearn.cluster import DBSCAN

Pancreas endocrinogenesis dataset [A. Bastidas-Ponce, S. Tritschler et al., 2019](https://github.com/theislab/pancreatic-endocrinogenesis/tree/master)

Pancreatic epithelial and Ngn3-Venus fusion (NVF) cells during secondary transition with transcriptome profiles sampled from embryonic day 15.5.

Endocrine cells are derived from endocrine progenitors located in the pancreatic epithelium. Endocrine commitment terminates in four major fates: glucagon- producing α-cells, insulin-producing β-cells, somatostatin-producing δ-cells and ghrelin-producing ε-cells. [Paper]( https://doi.org/10.1242/dev.173849
)

In [4]:
# Navigate to the scripts folder and run `python endocrinogenesis.py`
root = Path.cwd().parent
cache_dir = utils._cache_file_folder(root, "data/Pancreas/cache")
hash_fname = "pipeline_bundle_5d6f526d04ee.joblib" ## enter the cached pipeline bundle stored in data/Pancreas
bundle = load(cache_dir / hash_fname)

embedding = bundle["embedding_collection"]["embedding"]
print("Shape of pancreas data UMAP: ", embedding.shape)

Shape of pancreas data UMAP:  (3696, 2)


In [5]:
hvg_genes = bundle["highly_variable_genes"]
hvg_genes = hvg_genes.index[hvg_genes].tolist()
len(hvg_genes)

4000

In [6]:
metadata = pd.DataFrame(bundle["adata_metadata"])
logNormData = pd.DataFrame(bundle["adata_collection"]["logNormal"].toarray(), index = metadata.index, columns = hvg_genes)
metadata = pd.concat([logNormData, metadata], axis = 1)
metadata

,4732440D04Rik,Ncoa2,Eya1,Sbspon,Ube2w,Mcm3,Fam135a,Adgrb3,Zfp451,Uggt1,...,Rbbp7,Ap1s2,Tmem27,Ace2,Uty,Ddx3y,Eif2s3y,Erdr1,clusters_coarse,clusters
index,,,,,,,,,,,,,,,,,,,,,
AAACCTGAGAGGGATA,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.00000,0.000000,...,0.000000,0.000000,0.812927,0.0,0.0,0.000000,0.000000,0.000000,Pre-endocrine,Pre-endocrine
AAACCTGAGCCTTGAT,0.0,0.000000,0.0,0.000000,0.0,1.259227,0.000000,0.0,0.00000,0.000000,...,0.986488,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,Ductal,Ductal
AAACCTGAGGCAATTA,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.00000,0.000000,...,0.000000,0.915027,0.915027,0.0,0.0,0.000000,0.000000,0.000000,Endocrine,Alpha
AAACCTGCATCATCCC,0.0,0.000000,0.0,0.536473,0.0,0.883739,0.536473,0.0,0.00000,0.536473,...,1.141000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.536473,Ductal,Ductal
AAACCTGGTAAGTGGC,0.0,0.000000,0.0,0.000000,0.0,0.780221,0.780221,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,Ngn3 high EP,Ngn3 high EP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAAGTGACATA,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.500354,0.0,0.00000,0.000000,...,1.081098,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,Pre-endocrine,Pre-endocrine
TTTGTCAAGTGTGGCA,0.0,0.000000,0.0,0.000000,0.0,1.235657,0.595284,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,Ngn3 high EP,Ngn3 high EP
TTTGTCAGTTGTTTGG,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.00000,0.000000,...,1.712382,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,Ductal,Ductal


In [7]:
metadata["clusters_coarse"] = metadata["clusters_coarse"].cat.codes.values
metadata["clusters"] = metadata["clusters"].cat.codes.values

Interactive and Static TDA

In [8]:
# Creating mapper
cover = CubicalCover(n_intervals=10, overlap_frac=0.3)
clusterer = DBSCAN()
filter_func = FunctionTransformer(lambda x: x)

pipe = make_mapper_pipeline(
    filter_func=filter_func, 
    cover=cover,
    clusterer=clusterer,
    verbose=False,
    n_jobs=1,
)

In [9]:
#MIP = MapperInteractivePlotter(pipe, data) buggy
color_data = metadata.drop(["clusters", "clusters_coarse"], inplace = False, axis = 1)
plot_interactive_mapper_graph(pipe, embedding, color_data=color_data)


In [11]:
# Static TDA on the endocrinogenesis UMAP
fig = plot_static_mapper_graph(pipe, embedding)
fig.show(config={'scrollZoom': True})